# Module 08 — Notebook 02: Region Growing Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_08_RL_For_Image_Segmentation/02_Region_Growing_Agent/region_growing_agent.ipynb)

---

## Learning Objectives

1. Formulate region growing segmentation as a Markov Decision Process.
2. Design state features from the current segmentation frontier.
3. Implement an RL agent that selects expansion, stopping, and seed-placement actions.
4. Train the agent to maximise IoU improvement at every step.
5. Visualize region growth step-by-step in an animation-style output.

In [ ]:
!pip install torch torchvision numpy matplotlib scikit-image

In [ ]:
# Download REAL open-source dataset for segmentation (TINY - under 200MB total)
import torchvision
import numpy as np

# CIFAR-10: 60,000 real photos (already cached if downloaded before)
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
real_images = [np.array(cifar10[i][0]) for i in range(500)]
print(f"✅ CIFAR-10: {len(real_images)} real photos loaded (32x32 RGB)")

# FashionMNIST: 60,000 real clothing images (30MB only!)
fashion = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True)
print(f"✅ FashionMNIST: {len(fashion)} real clothing images (28x28)")
print("   Classes: T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Boot")

# Generate segmentation masks from CIFAR-10 using simple thresholding
# (This gives us real images with automatic pseudo-masks - no large dataset needed!)
def generate_pseudo_masks(images, threshold=128):
    """Generate binary masks from real images using Otsu-like thresholding."""
    masks = []
    for img in images:
        gray = np.mean(img, axis=2)
        mask = (gray > np.median(gray)).astype(np.uint8)
        masks.append(mask)
    return masks

pseudo_masks = generate_pseudo_masks(real_images)
print(f"✅ Generated {len(pseudo_masks)} segmentation masks from real images")
print("   Total download: ~200MB (CIFAR-10 + FashionMNIST)")

## Deep Derivation: Region Growing Mathematics and Graph-Based Segmentation

### Step 1: Region Growing Predicate — Homogeneity Criterion
A pixel $p$ is added to region $R$ if:
$$|I(p) - \mu_R| < \tau$$

where $\mu_R = \frac{1}{|R|}\sum_{q \in R} I(q)$ is the region mean and $\tau$ is the threshold.

**Proof this is equivalent to minimizing within-region variance:**

The variance of region $R \cup \{p\}$ is minimized when $I(p)$ is close to $\mu_R$:
$$\sigma^2_{R \cup \{p\}} = \frac{1}{|R|+1}\left[\sum_{q \in R}(I(q) - \mu')^2 + (I(p) - \mu')^2\right]$$

This is small iff $|I(p) - \mu_R|$ is small. $\blacksquare$

### Step 2: Adaptive Threshold via Statistical Testing
Test if pixel $p$ belongs to region $R$ using hypothesis testing:
$$H_0: I(p) \sim \mathcal{N}(\mu_R, \sigma_R^2) \quad \text{vs} \quad H_1: I(p) \not\sim \mathcal{N}(\mu_R, \sigma_R^2)$$

**Test statistic:** $z = \frac{|I(p) - \mu_R|}{\sigma_R}$

**Decision:** Accept $H_0$ (add pixel) if $z < z_{\alpha/2}$ (e.g., $z_{0.025} = 1.96$ for 95% confidence).

**Proof this controls false positive rate:**

Under $H_0$: $P(z > z_{\alpha/2}) = \alpha$, so at most $\alpha$ fraction of non-region pixels are wrongly added. $\blacksquare$

### Step 3: Region Merging — Graph Cut Formulation
**Merging criterion (Felzenszwalb & Huttenlocher):**

Merge regions $R_1, R_2$ if:
$$\text{Diff}(R_1, R_2) < \text{MInt}(R_1, R_2)$$

where:
- $\text{Diff}(R_1, R_2) = \min_{p \in R_1, q \in R_2, (p,q) \in E} w(p,q)$ (minimum edge weight between regions)
- $\text{MInt}(R_1, R_2) = \min(\text{Int}(R_1) + \tau(R_1), \text{Int}(R_2) + \tau(R_2))$
- $\text{Int}(R) = \max_{(p,q) \in \text{MST}(R)} w(p,q)$ (internal difference)
- $\tau(R) = k/|R|$ (threshold function, larger for smaller regions)

**Proof $\tau(R) = k/|R|$ prevents over-segmentation:**

Small regions have large $\tau$, making merging easier. As region grows, $\tau$ decreases, making further merging harder — this is a natural stopping criterion. $\blacksquare$

### Step 4: Seed Selection — Optimal Placement
For $K$ seeds in image of size $H \times W$:

**Grid initialization:** spacing $S = \sqrt{HW/K}$

**Proof grid minimizes maximum distance to nearest seed:**

For uniform grid with spacing $S$: $d_{\max} = \frac{S\sqrt{2}}{2}$

For any other placement, by pigeonhole principle, some cell of size $S \times S$ has no seed, giving $d_{\max} \geq \frac{S\sqrt{2}}{2}$. $\blacksquare$

### Step 5: Region Growing Order — Priority Queue Analysis
Using priority queue ordered by similarity:
- Insert: $O(\log N)$
- Extract-min: $O(\log N)$
- Total for image with $N$ pixels: $O(N \log N)$

**Proof this produces optimal region boundaries:**

Greedy addition of most-similar pixels first is equivalent to Prim's algorithm on the pixel graph — which finds the minimum spanning tree.

For MST-based segmentation, this gives optimal single-linkage clustering. $\blacksquare$

### Step 6: Multi-Scale Region Growing
At scale $\sigma$: $I_\sigma = G_\sigma * I$ (Gaussian-smoothed image)

**Scale-space property:** If a region boundary exists at scale $\sigma$, it exists at all finer scales $\sigma' < \sigma$.

**Proof (causality of Gaussian scale-space):**

$G_\sigma * I$ satisfies the heat equation: $\frac{\partial I}{\partial \sigma} = \nabla^2 I$. New extrema (boundaries) are never created as $\sigma$ increases — they can only merge or disappear. $\blacksquare$

### Step 7: RL for Region Growing — Sequential Decision Making
**State:** current segmentation mask + frontier pixels + image features

**Actions:** $a_t \in \{\text{grow}(d) \text{ for } d \in \{\text{N,S,E,W,NE,NW,SE,SW}\}\} \cup \{\text{stop}, \text{new\_seed}\}$

**Reward:**
$$r_t = \Delta\text{IoU}(R_t, G_t) + \lambda \cdot \mathbb{1}[\text{boundary preserved}]$$

**Connection to RL:** Traditional region growing uses fixed thresholds; an RL agent learns WHEN to grow, WHEN to stop, and WHEN to start new regions — adapting its strategy based on local image content and global segmentation quality.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torch
from collections import defaultdict
import random
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
random.seed(42)
torch.manual_seed(42)

print('All imports successful.')

---
## 1. Mathematical Formulation: MDP for Region Growing

We define an MDP $\mathcal{M} = (\mathcal{S}, \mathcal{A}, R, T, \gamma)$ where the agent iteratively grows segmentation regions.

### State Space $\mathcal{S}$

At step $t$ the state encodes:

$$
s_t = \big(\bar{I}_{\text{region}},\; \sigma_{\text{region}},\; \bar{I}_{\text{boundary}},\; \sigma_{\text{boundary}},\; |\text{boundary}|/|\text{image}|,\; |\text{region}|/|\text{image}|,\; \Delta\bar{I}\big)
$$

where $\bar{I}$ denotes mean intensity, $\sigma$ is standard deviation, $|\text{boundary}|$ is the number of boundary pixels, $|\text{region}|$ is the current region size, and $\Delta\bar{I} = |\bar{I}_{\text{region}} - \bar{I}_{\text{boundary}}|$ is the intensity contrast.

### Action Space $\mathcal{A}$

$$
\mathcal{A} = \{\text{EXPAND},\; \text{STOP},\; \text{NEW\_SEED}\}
$$

- **EXPAND**: merge the most similar boundary pixel into the current region.
- **STOP**: finalise the current region and move on.
- **NEW_SEED**: place a new seed at the unlabelled pixel with highest gradient magnitude.

### Reward Function $R$

The reward is the **IoU improvement** from step $t$ to $t+1$:

$$
r_t = \text{mIoU}(\hat{Y}^{(t+1)}, Y) - \text{mIoU}(\hat{Y}^{(t)}, Y)
$$

We also add a small penalty $-0.001$ per step to encourage efficiency.

### Transition $T$

Transitions are deterministic and depend on which action is executed:

$$
T(s_t, a) = \begin{cases}
s_{t+1}^{\text{expand}} & a = \text{EXPAND} \\
s_{t+1}^{\text{new\_region}} & a = \text{STOP or NEW\_SEED}
\end{cases}
$$

### Q-Learning Update

$$
Q(s,a) \leftarrow Q(s,a) + \alpha\big[r + \gamma \max_{a'}Q(s',a') - Q(s,a)\big]
$$

---
## 2. Synthetic Dataset

In [ ]:
def create_synthetic_image(size=64):
    img = np.zeros((size, size), dtype=np.float32)
    gt = np.zeros((size, size), dtype=np.int64)

    cy, cx, r = size // 4, size // 4, size // 6
    yy, xx = np.ogrid[:size, :size]
    circle = (yy - cy)**2 + (xx - cx)**2 <= r**2
    img[circle] = 0.8
    gt[circle] = 1

    sy, sx, sh = 3 * size // 5, 3 * size // 5, size // 7
    img[sy-sh:sy+sh, sx-sh:sx+sh] = 0.5
    gt[sy-sh:sy+sh, sx-sh:sx+sh] = 2

    img += np.random.normal(0, 0.03, img.shape).astype(np.float32)
    img = np.clip(img, 0, 1)
    return img, gt

IMG_SIZE = 48
image, ground_truth = create_synthetic_image(IMG_SIZE)
NUM_CLASSES = 3

cmap_gt = mcolors.ListedColormap(['black', 'dodgerblue', 'orange'])
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(image, cmap='gray'); axes[0].set_title('Synthetic Image'); axes[0].axis('off')
axes[1].imshow(ground_truth, cmap=cmap_gt, vmin=0, vmax=2); axes[1].set_title('Ground Truth'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## Deep Derivation: Maximum Flow / Minimum Cut Theorem and Graph-Based Segmentation

### Step 1: Image as Weighted Graph

Model an image as a graph $G = (V, E, w)$:
- $V$: set of pixels
- $E$: edges between adjacent pixels (4-connected or 8-connected)
- $w(p, q) = \exp\left(-\frac{|I(p) - I(q)|^2}{2\sigma^2}\right)$: edge weight (similarity)

### Step 2: Max-Flow Min-Cut Theorem (Ford-Fulkerson)

**Theorem:** In any flow network, the maximum flow from source $s$ to sink $t$ equals the minimum capacity of any $s$-$t$ cut.

$$\max_f |f| = \min_{(S,T)} \sum_{u \in S, v \in T} c(u,v)$$

**Proof sketch (LP duality):**

Max-flow is a linear program: $\max \sum_{v} f(s,v)$ subject to capacity and conservation constraints.

Its dual is: $\min \sum_{(u,v)} c(u,v) d(u,v)$ where $d(u,v) \in \{0,1\}$ indicates if edge $(u,v)$ crosses the cut.

By strong duality of LP, max-flow = min-cut. $\blacksquare$

### Step 3: Binary Segmentation as Minimum Cut

For foreground/background segmentation with additional unary terms:

$$E(\mathbf{x}) = \sum_p D_p(x_p) + \lambda \sum_{(p,q) \in E} V_{pq}(x_p, x_q)$$

- $D_p(0)$: cost of assigning pixel $p$ to background (connected to source)
- $D_p(1)$: cost of assigning pixel $p$ to foreground (connected to sink)
- $V_{pq}$: pairwise smoothness cost (the edge weights)

**Graph construction:** Add source $s$ and sink $t$. Edge capacities:
- $s \to p$: $D_p(1)$ (foreground likelihood)
- $p \to t$: $D_p(0)$ (background likelihood)
- $p \leftrightarrow q$: $\lambda V_{pq}$ (smoothness)

The minimum $s$-$t$ cut gives the optimal segmentation.

### Step 4: Normalized Cuts (Ncut) — Generalized Eigenvalue Problem

**Normalized cut** prevents trivial cuts (isolating single pixels):

$$\text{Ncut}(A, B) = \frac{\text{cut}(A,B)}{\text{assoc}(A,V)} + \frac{\text{cut}(A,B)}{\text{assoc}(B,V)}$$

where $\text{assoc}(A,V) = \sum_{i \in A, j \in V} w_{ij}$.

**Derivation as eigenvalue problem:**

Let $\mathbf{x} \in \{-1, +1\}^N$ be the indicator vector. Define:
- $\mathbf{W}$: weight matrix, $W_{ij} = w(i,j)$
- $\mathbf{D}$: degree matrix, $D_{ii} = \sum_j W_{ij}$
- $\mathbf{L} = \mathbf{D} - \mathbf{W}$: graph Laplacian

Then Ncut can be rewritten as:

$$\text{Ncut} = \frac{\mathbf{y}^T \mathbf{L} \mathbf{y}}{\mathbf{y}^T \mathbf{D} \mathbf{y}}$$

where $\mathbf{y} = (1 + \mathbf{x}) - b(1 - \mathbf{x})$ with $b = \text{assoc}(A,V)/\text{assoc}(B,V)$.

Minimizing this is a **generalized eigenvalue problem**:

$$\mathbf{L}\mathbf{y} = \lambda \mathbf{D}\mathbf{y}$$

The second-smallest eigenvector (Fiedler vector) gives the optimal partition. $\blacksquare$

### Step 5: Connection to RL Region Growing

Our RL agent makes **local, greedy decisions** (expand/stop/new seed) while graph cuts find **global optima**. The trade-offs:

| Approach | Complexity | Optimality | Adaptability |
|----------|-----------|------------|-------------|
| Min-cut | $O(N^2 \sqrt{N})$ | Global | Fixed cost function |
| Ncut | $O(N^{3/2})$ (Lanczos) | Approx. global | Fixed affinity |
| RL agent | $O(N)$ per episode | Learned policy | Adapts to image content |

The RL agent compensates for local decision-making by learning when to stop (avoiding over-segmentation) and when to seed new regions (avoiding under-segmentation).

---
## 3. Region Growing Environment

In [ ]:
class RegionGrowingEnv:
    """Environment where an RL agent grows segmentation regions."""
    EXPAND = 0
    STOP = 1
    NEW_SEED = 2

    def __init__(self, image, ground_truth, num_classes=3):
        self.image = image
        self.gt = ground_truth
        self.h, self.w = image.shape
        self.num_classes = num_classes
        self.reset()

    def reset(self):
        self.seg_map = -np.ones((self.h, self.w), dtype=np.int64)  # -1 = unlabelled
        self.current_label = 0
        self.current_region = set()
        self.boundary = set()
        self.step_count = 0
        self.snapshots = []
        self._place_seed(self._find_seed())
        return self._get_state()

    def _find_seed(self):
        unlabelled = np.argwhere(self.seg_map == -1)
        if len(unlabelled) == 0:
            return None
        grads = np.gradient(self.image)
        mag = np.sqrt(grads[0]**2 + grads[1]**2)
        best_idx = None
        best_val = -1
        for idx in range(0, len(unlabelled), max(1, len(unlabelled)//50)):
            r, c = unlabelled[idx]
            if mag[r, c] < 0.1:
                val = 1.0 / (mag[r, c] + 0.01)
                if val > best_val:
                    best_val = val
                    best_idx = (r, c)
        if best_idx is None:
            idx = np.random.randint(len(unlabelled))
            best_idx = tuple(unlabelled[idx])
        return best_idx

    def _place_seed(self, pos):
        if pos is None:
            return
        r, c = pos
        self.current_region = {(r, c)}
        self.seg_map[r, c] = self.current_label
        self._update_boundary()

    def _update_boundary(self):
        self.boundary = set()
        for (r, c) in self.current_region:
            for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                nr, nc = r+dr, c+dc
                if 0 <= nr < self.h and 0 <= nc < self.w and self.seg_map[nr, nc] == -1:
                    self.boundary.add((nr, nc))

    def _get_state(self):
        region_pixels = [self.image[r, c] for r, c in self.current_region] if self.current_region else [0.0]
        boundary_pixels = [self.image[r, c] for r, c in self.boundary] if self.boundary else [0.0]
        mean_r, std_r = np.mean(region_pixels), np.std(region_pixels) + 1e-6
        mean_b, std_b = np.mean(boundary_pixels), np.std(boundary_pixels) + 1e-6
        total = self.h * self.w
        state = (
            round(mean_r, 2), round(std_r, 2),
            round(mean_b, 2), round(std_b, 2),
            round(len(self.boundary) / total, 3),
            round(len(self.current_region) / total, 3),
            round(abs(mean_r - mean_b), 2)
        )
        return state

    def _compute_miou(self):
        pred = self.seg_map.copy()
        pred[pred == -1] = 0
        ious = []
        for k in range(self.num_classes):
            inter = np.sum((pred == k) & (self.gt == k))
            union = np.sum((pred == k) | (self.gt == k))
            ious.append(inter / union if union > 0 else 1.0)
        return np.mean(ious)

    def step(self, action):
        self.step_count += 1
        old_iou = self._compute_miou()
        done = False

        if action == self.EXPAND and self.boundary:
            region_mean = np.mean([self.image[r, c] for r, c in self.current_region])
            best_pixel = min(self.boundary, key=lambda p: abs(self.image[p[0], p[1]] - region_mean))
            r, c = best_pixel
            self.current_region.add((r, c))
            self.seg_map[r, c] = self.current_label
            self._update_boundary()

        elif action == self.STOP or action == self.NEW_SEED:
            self.current_label = min(self.current_label + 1, self.num_classes - 1)
            seed = self._find_seed()
            if seed is None:
                done = True
            else:
                self.current_region = set()
                self._place_seed(seed)

        elif action == self.EXPAND and not self.boundary:
            seed = self._find_seed()
            if seed is None:
                done = True
            else:
                self.current_label = min(self.current_label + 1, self.num_classes - 1)
                self.current_region = set()
                self._place_seed(seed)

        new_iou = self._compute_miou()
        reward = (new_iou - old_iou) * 10.0 - 0.001

        if np.sum(self.seg_map == -1) == 0:
            done = True

        if self.step_count > self.h * self.w:
            done = True

        if self.step_count % 50 == 0 or done:
            self.snapshots.append(self.seg_map.copy())

        return self._get_state(), reward, done

---
## 4. Q-Learning Agent

In [ ]:
class RegionGrowingAgent:
    def __init__(self, num_actions=3, alpha=0.2, gamma=0.95, epsilon=1.0, eps_decay=0.99, eps_min=0.05):
        self.num_actions = num_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.eps_decay = eps_decay
        self.eps_min = eps_min
        self.q_table = defaultdict(lambda: np.zeros(num_actions))

    def select_action(self, state):
        if np.random.random() < self.epsilon:
            return np.random.randint(self.num_actions)
        return int(np.argmax(self.q_table[state]))

    def update(self, state, action, reward, next_state, done):
        target = reward if done else reward + self.gamma * np.max(self.q_table[next_state])
        self.q_table[state][action] += self.alpha * (target - self.q_table[state][action])

    def decay_epsilon(self):
        self.epsilon = max(self.eps_min, self.epsilon * self.eps_decay)

## Deep Derivation: Statistical Region Merging and Hypothesis Testing

### Step 1: Statistical Region Merging (SRM) Framework

Nock and Nielsen (2004) proposed testing whether two adjacent regions should merge using order statistics.

**Null hypothesis:** Regions $R_1$ and $R_2$ have the same mean: $H_0: \mu_{R_1} = \mu_{R_2}$

**Merge predicate:**

$$|R_1 \text{ and } R_2 \text{ merge}| \iff |\bar{I}_{R_1} - \bar{I}_{R_2}| \leq \sqrt{b(R_1) + b(R_2)}$$

where the threshold function is:

$$b(R) = g^2 \cdot \frac{\ln(6N^2/\delta)}{2|R|}$$

with $g = 256/Q$ (quantization level), $N$ = number of pixels, and $\delta$ = significance level.

### Step 2: Derivation via Hoeffding's Inequality

**Hoeffding's inequality:** For bounded random variables $X_i \in [a, b]$ with mean $\mu$:

$$P\left(|\bar{X} - \mu| \geq t\right) \leq 2\exp\left(-\frac{2n t^2}{(b-a)^2}\right)$$

**Application to SRM:** For pixel intensities in $[0, g]$ with true mean $\mu_R$:

$$P\left(|\bar{I}_R - \mu_R| \geq \epsilon\right) \leq 2\exp\left(-\frac{2|R|\epsilon^2}{g^2}\right)$$

Setting the right side to $\delta/(3N^2)$ (Bonferroni correction for $N^2$ possible pairs):

$$\epsilon = g\sqrt{\frac{\ln(6N^2/\delta)}{2|R|}} = \sqrt{b(R)}$$

By the union bound over both regions:

$$P\left(|\bar{I}_{R_1} - \bar{I}_{R_2}| \geq \sqrt{b(R_1)} + \sqrt{b(R_2)} \mid \mu_{R_1} = \mu_{R_2}\right) \leq \delta/N^2$$

This controls the false merge rate over the entire image. $\blacksquare$

### Step 3: Felzenszwalb Segmentation Criterion (Revisited)

**Internal difference:** $\text{Int}(R) = \max_{(p,q) \in \text{MST}(R)} w(p,q)$

**Minimum internal difference:** $\text{MInt}(R_1, R_2) = \min(\text{Int}(R_1) + \tau(R_1), \text{Int}(R_2) + \tau(R_2))$

**Merge condition:** $\text{Diff}(R_1, R_2) < \text{MInt}(R_1, R_2)$

**Proof this produces neither too fine nor too coarse segmentation:**

The threshold $\tau(R) = k/|R|$ adapts to region size. For small regions ($|R|$ small), $\tau$ is large → easy to merge → avoids over-segmentation. For large regions ($|R|$ large), $\tau$ is small → hard to merge → avoids under-segmentation. The parameter $k$ controls this balance. $\blacksquare$

### Step 4: Region Adjacency Graph (RAG) Merging Complexity

The RAG has:
- Vertices: current regions
- Edges: adjacency relationships

**Merging procedure complexity:**

1. Build initial RAG: $O(N)$ (scan all pixels and their neighbors)
2. Each merge: $O(\text{degree})$ (update adjacency of merged region)
3. Total merges: at most $N - 1$ (from $N$ regions to 1)
4. **Total complexity: $O(N \log N)$** (with priority queue for merge ordering)

### Step 5: RL Agent as Learned Merge Oracle

Our RL agent learns **when to merge (EXPAND)** and **when to stop**, implicitly learning the merge predicate. The Q-values encode:

$$Q(s, \text{EXPAND}) > Q(s, \text{STOP}) \iff \text{predicted IoU gain from expanding} > 0$$

This is a data-driven alternative to the fixed statistical thresholds in SRM, adapting the merge criterion to the specific image content and segmentation objective.

---
## 5. Training Loop

In [ ]:
def train_region_agent(image, ground_truth, num_episodes=40, max_steps=600):
    agent = RegionGrowingAgent(num_actions=3, alpha=0.2, gamma=0.95, epsilon=1.0, eps_decay=0.95, eps_min=0.05)
    env = RegionGrowingEnv(image, ground_truth)
    history = {'episode': [], 'miou': [], 'reward': [], 'steps': []}
    best_snapshots = None
    best_iou = -1

    for ep in range(num_episodes):
        state = env.reset()
        total_reward = 0.0

        for step in range(max_steps):
            action = agent.select_action(state)
            next_state, reward, done = env.step(action)
            agent.update(state, action, reward, next_state, done)
            state = next_state
            total_reward += reward
            if done:
                break

        agent.decay_epsilon()
        final_iou = env._compute_miou()
        history['episode'].append(ep)
        history['miou'].append(final_iou)
        history['reward'].append(total_reward)
        history['steps'].append(step + 1)

        if final_iou > best_iou:
            best_iou = final_iou
            best_snapshots = env.snapshots.copy()

        if ep % 5 == 0:
            print(f'Episode {ep:3d} | mIoU: {final_iou:.4f} | Reward: {total_reward:7.2f} | Steps: {step+1} | ε: {agent.epsilon:.3f}')

    return agent, history, best_snapshots

agent, history, best_snapshots = train_region_agent(image, ground_truth, num_episodes=40)
print('\nTraining complete.')

---
## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history['episode'], history['miou'], 'o-', color='teal', markersize=3)
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('mIoU'); axes[0].set_title('Segmentation Quality')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['episode'], history['reward'], 'o-', color='coral', markersize=3)
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Total Reward'); axes[1].set_title('Cumulative Reward')
axes[1].grid(True, alpha=0.3)

axes[2].plot(history['episode'], history['steps'], 'o-', color='purple', markersize=3)
axes[2].set_xlabel('Episode'); axes[2].set_ylabel('Steps'); axes[2].set_title('Steps per Episode')
axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

---
## 7. Step-by-Step Region Growth Visualization

In [ ]:
if best_snapshots and len(best_snapshots) > 0:
    n_show = min(8, len(best_snapshots))
    indices = np.linspace(0, len(best_snapshots)-1, n_show, dtype=int)
    fig, axes = plt.subplots(1, n_show + 1, figsize=(3 * (n_show + 1), 3))

    axes[0].imshow(ground_truth, cmap=cmap_gt, vmin=0, vmax=2)
    axes[0].set_title('Ground Truth'); axes[0].axis('off')

    cmap_seg = mcolors.ListedColormap(['lightgray', 'dodgerblue', 'orange', 'green'])

    for plot_idx, snap_idx in enumerate(indices):
        snap = best_snapshots[snap_idx].copy()
        display = snap.copy()
        display[display == -1] = 3  # unlabelled -> green tint
        axes[plot_idx + 1].imshow(display, cmap=cmap_seg, vmin=0, vmax=3)
        labelled_pct = 100 * np.sum(snap >= 0) / snap.size
        axes[plot_idx + 1].set_title(f'Step {snap_idx*50}\n{labelled_pct:.0f}% labelled')
        axes[plot_idx + 1].axis('off')

    plt.suptitle('Region Growth Progression (Best Episode)', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print('No snapshots recorded.')

## Deep Derivation: Region Growing Optimality and Markov Random Fields

### Step 1: Segmentation as MAP Estimation on MRF

Model the label field $\mathbf{Y}$ as a Markov Random Field:

$$P(\mathbf{Y} | \mathbf{I}) \propto \exp\left(-\sum_p \psi_p(y_p, I_p) - \lambda \sum_{(p,q) \in \mathcal{N}} \psi_{pq}(y_p, y_q)\right)$$

**Unary potential:** $\psi_p(y_p, I_p) = -\log P(I_p | y_p)$ (class likelihood)

**Pairwise potential:** $\psi_{pq}(y_p, y_q) = [y_p \neq y_q] \cdot w_{pq}$ (Potts model, smoothness)

$$w_{pq} = \exp\left(-\frac{|I_p - I_q|^2}{2\sigma^2}\right) \cdot \frac{1}{\|p - q\|}$$

### Step 2: Connection to Region Growing

Region growing with threshold $\tau$ is equivalent to MAP estimation with:

$$\psi_p(k) = -\frac{(I_p - \mu_k)^2}{2\sigma_k^2}, \quad \psi_{pq}(y_p, y_q) = [y_p \neq y_q]$$

**Proof:** The grow condition $|I(p) - \mu_R| < \tau$ is equivalent to:

$$\psi_p(k) = -\frac{(I_p - \mu_k)^2}{2\sigma^2} > -\frac{\tau^2}{2\sigma^2}$$

i.e., the unary likelihood exceeds a threshold. The pairwise term prevents label changes unless the intensity difference is large. $\blacksquare$

### Step 3: Greedy vs. Global Optimization

**Region growing = greedy:** At each step, add the best pixel. This is suboptimal in general.

**Theorem (submodularity):** If the segmentation quality function $f$ is submodular (diminishing returns), greedy achieves at least $(1 - 1/e)$ fraction of the optimal.

**IoU is NOT submodular in general**, so region growing may not have approximation guarantees. However, empirically it works well because:

1. Natural images have smooth intensity fields → local decisions are often globally consistent
2. The RL agent learns to compensate for greedy myopia through the discount factor $\gamma$

### Step 4: Bellman Equation for Region Growing

For our MDP, the optimal value function satisfies:

$$V^*(s) = \max_{a \in \{\text{EXPAND, STOP, NEW\_SEED}\}} \left[R(s, a) + \gamma \sum_{s'} T(s' | s, a) V^*(s')\right]$$

Since transitions are deterministic:

$$V^*(s) = \max_a \left[R(s, a) + \gamma V^*(s'(s, a))\right]$$

**Interpretation:** $V^*(s)$ represents the maximum future IoU improvement achievable from state $s$. The agent expands when $Q(s, \text{EXPAND}) > Q(s, \text{STOP})$, i.e., when continued growing is expected to improve segmentation quality more than stopping. $\blacksquare$

---
## 8. Animation-Style Frame Sequence

In [ ]:
def generate_animation_frames(image, ground_truth, agent, max_steps=600):
    """Run greedy policy and capture detailed frames."""
    env = RegionGrowingEnv(image, ground_truth)
    state = env.reset()
    frames = [env.seg_map.copy()]

    for step in range(max_steps):
        action = int(np.argmax(agent.q_table[state]))
        state, _, done = env.step(action)
        if step % 20 == 0:
            frames.append(env.seg_map.copy())
        if done:
            frames.append(env.seg_map.copy())
            break
    return frames

frames = generate_animation_frames(image, ground_truth, agent)

n_frames = min(10, len(frames))
sel = np.linspace(0, len(frames)-1, n_frames, dtype=int)
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
cmap_seg = mcolors.ListedColormap(['lightgray', 'dodgerblue', 'orange', 'green'])

for plot_idx, frame_idx in enumerate(sel):
    ax = axes[plot_idx // 5, plot_idx % 5]
    f = frames[frame_idx].copy()
    f[f == -1] = 3
    ax.imshow(f, cmap=cmap_seg, vmin=0, vmax=3)
    ax.set_title(f'Frame {frame_idx}', fontsize=10)
    ax.axis('off')

plt.suptitle('Animation Frames: Region Growing Agent', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 9. Final Comparison

In [ ]:
env_eval = RegionGrowingEnv(image, ground_truth)
state = env_eval.reset()
for step in range(800):
    action = int(np.argmax(agent.q_table[state]))
    state, _, done = env_eval.step(action)
    if done:
        break

final_seg = env_eval.seg_map.copy()
final_seg[final_seg == -1] = 0
final_iou = env_eval._compute_miou()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(image, cmap='gray'); axes[0].set_title('Input Image'); axes[0].axis('off')
axes[1].imshow(ground_truth, cmap=cmap_gt, vmin=0, vmax=2); axes[1].set_title('Ground Truth'); axes[1].axis('off')
axes[2].imshow(final_seg, cmap=cmap_gt, vmin=0, vmax=2); axes[2].set_title(f'Agent Result (mIoU={final_iou:.3f})'); axes[2].axis('off')
plt.suptitle('Region Growing: Final Segmentation', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

for k, name in enumerate(['Background', 'Circle', 'Square']):
    inter = np.sum((final_seg == k) & (ground_truth == k))
    union = np.sum((final_seg == k) | (ground_truth == k))
    iou_k = inter / union if union > 0 else 1.0
    print(f'  Class {k} ({name:>10s}): IoU = {iou_k:.4f}')
print(f'  Mean IoU: {final_iou:.4f}')

---
## Summary

In this notebook we:

1. **Formulated region growing as an MDP** with three actions (EXPAND, STOP, NEW_SEED) and IoU-improvement rewards.
2. **Designed compact state features** capturing region statistics, boundary contrast, and relative sizes.
3. **Trained a tabular Q-learning agent** to learn when to expand, stop, or start new regions.
4. **Visualized step-by-step region growth** showing how regions emerge from seeds.
5. **Compared final segmentation** with the ground truth, demonstrating the agent's ability to segment distinct objects.

### Key Takeaways

| Concept | Detail |
|---|---|
| State | Region/boundary statistics (7-dim) |
| Actions | EXPAND / STOP / NEW_SEED |
| Reward | $\Delta$mIoU per step $- 0.001$ |
| Algorithm | Tabular Q-learning |
| Visualisation | Frame-by-frame region growth |

**Next →** Notebook 03 implements a **Boundary Detection Agent** that traces object edges.